## 1. Project introduction

Phase 10 uses persisted detector artifacts and their fitted preprocessors. No production model is trained here.

## 2. Explainable AI objectives

Explain model outputs with explicit units, deterministic reference data, bounded samples, and record-level provenance.

## 3. SHAP methodology

SHAP values are relative to a model output, background data, and perturbation assumptions; they are not causal claims.

## 4. Existing model artifact inspection

Inspect configured artifact identifiers, versions, model families, output spaces, and artifact paths.

In [ ]:
from pathlib import Path
import json, tomllib
config = tomllib.loads(Path("configs/explainability.toml").read_text(encoding="utf-8"))
config["models"]

## 5. Preprocessing and feature schema inspection

Validate the original feature contract and exact saved transformed feature names.

In [ ]:
from intrusion_intelligence.feature_engineering import FeatureContract
from intrusion_intelligence.explainability.model_registry import load_explainable_models
contract = FeatureContract.from_file(config["dataset"]["feature_schema"])
models, _ = load_explainable_models("configs/explainability.toml", contract=contract)
{key: {"model_id": model.model_id, "features": len(model.feature_names), "outputs": [out.name for out in model.outputs]} for key, model in models.items()}

## 6. Background dataset preparation

Use only training partition rows; anomaly reference rows are BENIGN training observations.

In [ ]:
from intrusion_intelligence.explainability.background_data import load_training_background
dataset = Path(config["dataset"]["parquet_directory"])
{key: "loaded during the bounded pipeline run; see report background metadata" for key in models}

## 7. Explainer compatibility investigation

Use TreeExplainer for supported forest outputs, LinearExplainer for Logistic Regression margins/logits, and PermutationExplainer for the transformed Isolation Forest score.

## 8. Binary classification global explanations

Review global mean absolute SHAP ranks for ATTACK score or ATTACK log-odds, depending on model.

## 9. Binary classification local explanations

Inspect signed contributions and the output reconstruction for selected Phase 09 flows.

## 10. Multi-class global explanations

Review all taxonomy-associated class outputs with saved class labels.

## 11. Class-specific explanations

Compare configured output labels and their output semantics; class IDs come from persisted taxonomy.

## 12. Multi-class local explanations

Review selected per-class output contributions for the referenced flow.

## 13. Anomaly detector SHAP compatibility

Permutation SHAP wraps exactly negative score_samples so higher scores mean more anomalous.

## 14. Global anomaly explanations

Inspect bounded mean absolute attribution for the Isolation Forest anomaly score.

## 15. Local anomaly explanations

Read per-flow signed effects relative to the benign-only training background and calibrated threshold.

## 16. Original feature mapping

Review grouped signed local contributions and global sums of transformed mean absolute values; unmapped features stay explicit.

## 17. SHAP visualization

Inspect generated bar and beeswarm plots in the run figure directory.

## 18. Model explanation comparison

Compare explanations only within the same output scale and compatible background methodology.

## 19. Threat intelligence alert enrichment

Inspect sidecar explanation references; threat scores, severity, and correlation evidence remain unchanged.

## 20. Explanation report generation

Read JSONL records, CSV global rankings, JSON metadata, Markdown report, and figure outputs.

In [ ]:
# This runs all configured bounded model/output explanations and writes timestamped reports.
from intrusion_intelligence.explainability.pipeline import run_explainability
result = run_explainability("configs/explainability.toml")
result

## 21. Interpretation limitations

Correlated inputs, benchmark shift, background choices, and approximate model-agnostic explanations limit interpretation.

## 22. Findings and recommendations

Record measured output semantics, validation residuals, unsupported combinations, and open caveats without inferring causality.

In [ ]:
report_path = Path(result["output_directory"]) / "explainability_report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
{ "run_id": report["run_id"], "models": list(report["models"]), "explanation_records": report["explanation_records"], "limitations": report["limitations"] }